In [0]:
from pyspark.sql.functions import *

spark.conf.set(
"fs.azure.account.key.ecomdatalakegen2.dfs.core.windows.net",
dbutils.secrets.get(
    scope="azure",
    key="storage-key"
))

storage_account = "ecomdatalakegen2"
silver_base = f"abfss://silver@{storage_account}.dfs.core.windows.net"

# Create database if it doesn't exist
spark.sql("CREATE DATABASE IF NOT EXISTS ecom_gold")
spark.sql("USE ecom_gold")

# Read Silver
customers = spark.read.format("delta").load(f"{silver_base}/customers")
products   = spark.read.format("delta").load(f"{silver_base}/products")
orders     = spark.read.format("delta").load(f"{silver_base}/orders")
payments   = spark.read.format("delta").load(f"{silver_base}/payments")
reviews    = spark.read.format("delta").load(f"{silver_base}/reviews")
clickstream = spark.read.format("delta").load(f"{silver_base}/clickstream")





In [0]:
# ----- Dimensions -----
customers.select("customer_id","name","email","city","state","signup_date") \
    .write.format("delta").mode("overwrite").saveAsTable("ecom_gold.dim_customer")

products.select("product_id", col("name").alias("product_name"),"category","price","cost") \
    .write.format("delta").mode("overwrite").saveAsTable("ecom_gold.dim_product")

dates = (orders.select(col("order_date").alias("dt"))
             .union(clickstream.select(col("timestamp").alias("dt")))
             .select(to_date("dt").alias("date"))
             .distinct())
dim_date = dates.select(
    date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    "date",
    year("date").alias("year"),
    month("date").alias("month"),
    dayofmonth("date").alias("day")
)
dim_date.write.format("delta").mode("overwrite").saveAsTable("ecom_gold.dim_date")

payments.select("payment_method").distinct() \
    .withColumn("payment_method_id", monotonically_increasing_id()) \
    .write.format("delta").mode("overwrite").saveAsTable("ecom_gold.dim_payment_method")

In [0]:
from pyspark.sql.functions import date_trunc
# ----- Facts -----
dim_date_df = spark.table("ecom_gold.dim_date")
payment_method_df = spark.table("ecom_gold.dim_payment_method")

fact_orders = orders.alias("o") \
    .join(dim_date_df.alias("d"), to_date("o.order_date") == col("d.date")) \
    .join(payments.alias("p"), "order_id") \
    .join(payment_method_df.alias("pm"), col("p.payment_method") == col("pm.payment_method")) \
    .select(
        col("o.order_id"), col("o.customer_id"), col("o.product_id"),
        col("d.date_key"), col("pm.payment_method_id"),
        col("o.quantity"), col("o.price"),
        (col("o.quantity") * col("o.price")).alias("total_amount"),
        lit(0.0).alias("discount")
    )
fact_orders.write.format("delta").mode("overwrite").saveAsTable("ecom_gold.fact_orders")

fact_reviews = reviews.alias("r") \
    .join(dim_date_df.alias("d"), to_date("r.review_date") == col("d.date")) \
    .select("r.review_id", "r.customer_id", "r.product_id", "d.date_key", "r.rating", "r.review_text")
fact_reviews.write.format("delta").mode("overwrite").saveAsTable("ecom_gold.fact_reviews")

fact_clickstream = clickstream.alias("c") \
    .join(dim_date_df.alias("d"), to_date("c.timestamp") == col("d.date")) \
    .select("c.event_id", "c.session_id", "c.customer_id", "c.product_id",
            "d.date_key", "c.event_type", "c.timestamp")
fact_clickstream = fact_clickstream.withColumn("event_minute", date_trunc("minute", col("timestamp")))
fact_clickstream.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("ecom_gold.fact_clickstream")
